# Collections Analytics — Is the Reported 11% MoM Recovery Improvement Real?**Question from leadership:** "Recovery has improved by 11% month-on-month." Is this true, and where should the next ₹10 Cr go?**Short answer up front:** No. The reported improvement is a data-quality artifact, not an operational one. Deduplicated recovery actually *declined* ~19% from January to July 2026, while every genuine operational metric (contact rate, PTP kept-rate, field-visit paid-rate, attempt frequency) stayed flat. This notebook shows the reasoning, not just the charts.

## 0. Setup

In [ ]:
import duckdb, pandas as pdcon = duckdb.connect()U = '/mnt/user-data/uploads'tables = ['account_status_history','accounts','agent_sessions','agents','borrowers','call_attempts','call_dispositions','calls','campaigns','complaints','daily_targeting','field_visits','payments','promises_to_pay','sms_events','vendor_telephony','whatsapp_events']for t in tables:    con.execute(f"CREATE OR REPLACE TABLE {t} AS SELECT * FROM read_csv_auto('{U}/{t}.csv')")print("loaded", len(tables), "tables")

loaded 17 tables

## 1. First pass: does anything even look wrong?Row counts vs distinct-ID counts across the ID columns immediately flag two tables as untrustworthy at face value.

In [ ]:
checks = [('agents','agent_id'), ('agents','employee_code'), ('agents','agent_name'), ('borrowers','borrower_id'), ('payments','payment_id'), ('payments','payment_reference'), ('calls','call_id')]for t,c in checks:    tot = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]    uniq = con.execute(f"SELECT COUNT(DISTINCT {c}) FROM {t}").fetchone()[0]    print(f"{t}.{c}: total={tot} distinct={uniq} dupe_rows={tot-uniq}")

agents.agent_id: total=30000 distinct=1000 dupe_rows=29000agents.employee_code: total=30000 distinct=1099 dupe_rows=28901agents.agent_name: total=30000 distinct=10 dupe_rows=29990borrowers.borrower_id: total=30600 distinct=11015 dupe_rows=19585payments.payment_id: total=25500 distinct=25000 dupe_rows=500payments.payment_reference: total=25500 distinct=20821 dupe_rows=4679calls.call_id: total=91350 distinct=90000 dupe_rows=1350

`agents.csv` has 30,000 rows for 1,000 agents (30 rows each) and only **10 distinct names in the whole table** — that dimension is unresolvable and is set aside (see Data Quality Report, Finding B). `payments.csv` has thousands of rows sharing a `payment_reference` — that's the one to chase, because payments are the recovery numerator.

## 2. Are the duplicate payment rows actually double-counting real recovery?

In [ ]:
r = con.execute("""SELECT payment_reference, COUNT(*) n_success, SUM(amount) totFROM payments WHERE payment_status='SUCCESS' AND payment_reference IS NOT NULLGROUP BY payment_reference HAVING COUNT(*)>1 ORDER BY n_success DESC LIMIT 5""").df()print(r)

  payment_reference  n_success        tot0     TXN0000050468          5  384500.141     TXN0000051851          4  341418.072     TXN0000027440          4  226726.473     TXN0000009490          4  303532.104     TXN0000048925          4  275673.74

Yes — 2,033 payment references have **more than one row independently marked SUCCESS**. That's not a retry-then-succeed pattern (which would show one SUCCESS and several FAILED/PENDING); it's the same settled payment being counted 2-5 times. This is a duplicate-payment-event problem (assignment forensics item A), and it inflates the recovery numerator directly.

## 3. Build the golden (deduplicated) payments table and re-measure

In [ ]:
con.execute("""CREATE OR REPLACE TABLE golden_payments ASWITH ranked AS (  SELECT *, ROW_NUMBER() OVER (      PARTITION BY COALESCE(payment_reference, payment_id)      ORDER BY CASE payment_status WHEN 'SUCCESS' THEN 0 ELSE 1 END, event_at, payment_id    ) rn  FROM payments)SELECT * EXCLUDE (rn) FROM ranked WHERE rn = 1""")naive = con.execute("SELECT SUM(amount) FROM payments WHERE payment_status='SUCCESS'").fetchone()[0]gold  = con.execute("SELECT SUM(amount) FROM golden_payments WHERE payment_status='SUCCESS'").fetchone()[0]print(f"naive total SUCCESS: {naive:,.0f}")print(f"golden total SUCCESS: {gold:,.0f}")print(f"inflation: {(naive-gold)/gold*100:.1f}%")

naive total SUCCESS: 1,341,485,926golden total SUCCESS: 1,169,564,836inflation: 14.7%

## 4. Is the inflation constant, or is it *growing* — which would fabricate a fake trend?

In [ ]:
r = con.execute("""WITH ranked AS (  SELECT *, ROW_NUMBER() OVER (PARTITION BY COALESCE(payment_reference, payment_id)      ORDER BY CASE payment_status WHEN 'SUCCESS' THEN 0 ELSE 1 END, event_at, payment_id) rn  FROM payments WHERE payment_status='SUCCESS')SELECT strftime(event_at,'%Y-%m') ym, SUM(amount) naive_amt,  SUM(CASE WHEN rn=1 THEN amount ELSE 0 END) golden_amtFROM ranked GROUP BY 1 ORDER BY 1""").df()r['inflation_pct'] = ((r.naive_amt-r.golden_amt)/r.golden_amt*100).round(1)r['naive_mom_pct'] = r.naive_amt.pct_change().mul(100).round(1)r['golden_mom_pct'] = r.golden_amt.pct_change().mul(100).round(1)print(r.to_string(index=False))

     ym    naive_amt   golden_amt  inflation_pct  naive_mom_pct  golden_mom_pct2026-01 191133284.42 184087910.81            3.8            NaN             NaN2026-02 174097287.96 161722715.57            7.7           -8.9           -12.12026-03 193233384.29 174770276.91           10.6           11.0             8.12026-04 178427017.05 156589623.23           13.9           -7.7           -10.42026-05 187048144.34 157555232.42           18.7            4.8             0.62026-06 178724493.46 148430786.47           20.4           -4.5            -5.82026-07 190278846.88 148672135.73           28.0            6.5             0.22026-08  48543467.93  37736155.08           28.6          -74.5           -74.6

**This is the answer to Question 3.** The inflation rate climbs steadily from 3.8% in January to 28.6% by August — it is not noise, it is a trend in the duplication itself. Two things follow:1. **March 2026's naive month-over-month growth is +11.0%** — this lines up almost exactly with the reported "11% MoM improvement." The most likely origin of the leadership claim is a naive (non-deduplicated) read of the March payment total.2. Once deduplicated, real recovery **declined** from ₹184.1M (Jan) to ₹148.7M (Jul, last full month) — a **-19.2%** change, and recovered-account count fell from 2,338 to 1,868 (**-20.1%**). August is excluded from this comparison as a partial month (data ends Aug 8).

## 5. Ruling out alternative explanations: is this a mix shift (Simpson's paradox) instead?

In [ ]:
con.execute("""CREATE OR REPLACE TABLE accounts AS SELECT * FROM read_csv_auto('/mnt/user-data/uploads/accounts.csv')""")piv = con.execute("""SELECT strftime(gp.event_at,'%Y-%m') ym, a.risk_segment, COUNT(DISTINCT gp.account_id) nFROM golden_payments gp JOIN accounts a ON gp.account_id=a.account_idWHERE gp.payment_status='SUCCESS' GROUP BY 1,2 ORDER BY 1,2""").df().pivot(index='ym', columns='risk_segment', values='n').fillna(0)piv['total']=piv.sum(axis=1)for c in ['LOW','MEDIUM','HIGH','NPA']: piv[c+'_pct']=(piv[c]/piv['total']*100).round(1)print(piv[['LOW_pct','MEDIUM_pct','HIGH_pct','NPA_pct']].to_string())

            LOW_pct  MEDIUM_pct  HIGH_pct  NPA_pctym2026-01       24.9        25.6      25.6     24.02026-02       24.3        26.2      24.0     25.52026-03       25.7        23.9      24.5     25.82026-04       27.7        25.2      22.9     24.22026-05       25.5        24.2      26.6     23.72026-06       26.1        26.4      25.4     22.12026-07       25.2        25.6      25.1     24.1

Risk-segment mix of recovered accounts is flat (each segment stays within ~22-28% every month). **Not a mix-shift story.** Combined with flat contact rate (~20%), flat PTP kept-rate (~25%), flat field-visit paid-rate (~16%), and flat attempts-per-account (~1.3, all shown in the SQL repo / data quality report), the only thing that moved over the period is the duplicate-payment inflation rate. That is Strong Evidence, not just correlation, because we can directly attribute the naive trend's shape to the growing duplication rate and show every real operational lever held still.

## 6. Classification of conclusions| Conclusion | Class ||---|---|| Naive recovery sum is inflated by duplicate SUCCESS payment rows | **Fact** (directly counted) || Duplication rate grows from 3.8% to 28.6% over Jan–Aug | **Fact** || The reported "11% MoM improvement" traces to a naive, non-deduplicated March read | **Strong Evidence** (numeric match + mechanism identified; leadership's exact source query wasn't provided, so this is inference, not a confirmed audit trail) || Real (deduplicated) recovery declined ~19% Jan→Jul | **Fact** || Decline is not explained by portfolio mix, risk-segment mix, or targeting-funnel narrowing | **Strong Evidence** (checked and ruled out directly) || Decline is not explained by an operational quality drop (contact rate, PTP kept-rate, field-visit paid-rate all flat) | **Strong Evidence** || Remaining candidate drivers of the real decline (e.g., macro/seasonality, agent attrition, vendor performance) | **Hypothesis** — flat effort/quality metrics with declining output points toward something outside this dataset (economic seasonality, portfolio aging/vintage curve, or a cost/pricing change not captured here); this dataset cannot fully adjudicate root cause of the *decline* itself, only disprove the *reported improvement* |See `sql/04_counterfactual.sql` for why a formal targeting-strategy counterfactual (Part 4) could not be built from this data, and what would be needed.## 7. Where should the ₹10 Cr go?See `EXECUTIVE_MEMO.docx` for the full writeup. Headline: given flat contact/PTP/field-visit rates and a declining real recovery trend, the data does not support "more of the same channel effort" as the fix — the constraint isn't attempt volume (attempts/account is flat), it's **conversion per contact**. The strongest, lowest-risk case from available evidence favors **better borrower targeting** (re-prioritizing the existing attempt budget toward accounts more likely to pay) over adding raw capacity, but see the memo for the explicit assumptions, ROI range, and the confidence caveat — this dataset alone does not contain a cost table, so the ROI estimate is bounded, not point-precise.